In [16]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd

books_path = "/content/drive/MyDrive/Colab Notebooks/amazon_reviews_us_Books_v1_02.tsv"

books_sample = pd.read_csv(
    books_path,
    sep="\t",
    nrows=1000
)

print("Shape of sample:", books_sample.shape)
print("\nColumns:")
print(books_sample.columns.tolist())
books_sample.head()

Shape of sample: (1000, 15)

Columns:
['marketplace', 'customer_id', 'review_id', 'product_id', 'product_parent', 'product_title', 'product_category', 'star_rating', 'helpful_votes', 'total_votes', 'vine', 'verified_purchase', 'review_headline', 'review_body', 'review_date']


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,12076615,RQ58W7SMO911M,0385730586,122662979,Sisterhood of the Traveling Pants (Book 1),Books,4,2,3,N,N,this book was a great learning novel!,this boook was a great one that you could lear...,2005-10-14
1,US,12703090,RF6IUKMGL8SF,0811828964,56191234,The Bad Girl's Guide to Getting What You Want,Books,3,5,5,N,N,Fun Fluff,If you are looking for something to stimulate ...,2005-10-14
2,US,12257412,R1DOSHH6AI622S,1844161560,253182049,"Eisenhorn (A Warhammer 40,000 Omnibus)",Books,4,1,22,N,N,this isn't a review,never read it-a young relative idicated he lik...,2005-10-14
3,US,50732546,RATOTLA3OF70O,0373836635,348672532,Colby Conspiracy (Colby Agency),Books,5,2,2,N,N,fine author on her A-game,Though she is honored to be Chicago Woman of t...,2005-10-14
4,US,51964897,R1TNWRKIVHVYOV,0262181533,598678717,The Psychology of Proof: Deductive Reasoning i...,Books,4,0,2,N,N,Execellent cursor examination,Review based on a cursory examination by Unive...,2005-10-14


In [18]:
electronics_path = "/content/drive/MyDrive/Colab Notebooks/amazon_reviews_us_Electronics_v1_00.tsv"

electronics_sample = pd.read_csv(
    electronics_path,
    sep="\t",
    nrows=1000
)

print("Shape of sample:", electronics_sample.shape)
print("\nColumns:")
print(electronics_sample.columns.tolist())
electronics_sample.head()

Shape of sample: (1000, 15)

Columns:
['marketplace', 'customer_id', 'review_id', 'product_id', 'product_parent', 'product_title', 'product_category', 'star_rating', 'helpful_votes', 'total_votes', 'vine', 'verified_purchase', 'review_headline', 'review_body', 'review_date']


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,41409413,R2MTG1GCZLR2DK,B00428R89M,112201306,yoomall 5M Antenna WIFI RP-SMA Female to Male ...,Electronics,5,0,0,N,Y,Five Stars,As described.,2015-08-31
1,US,49668221,R2HBOEM8LE9928,B000068O48,734576678,"Hosa GPM-103 3.5mm TRS to 1/4"" TRS Adaptor",Electronics,5,0,0,N,Y,It works as advertising.,It works as advertising.,2015-08-31
2,US,12338275,R1P4RW1R9FDPEE,B000GGKOG8,614448099,Channel Master Titan 2 Antenna Preamplifier,Electronics,5,1,1,N,Y,Five Stars,Works pissa,2015-08-31
3,US,38487968,R1EBPM82ENI67M,B000NU4OTA,72265257,LIMTECH Wall charger + USB Hotsync & Charging ...,Electronics,1,0,0,N,Y,One Star,Did not work at all.,2015-08-31
4,US,23732619,R372S58V6D11AT,B00JOQIO6S,308169188,Skullcandy Air Raid Portable Bluetooth Speaker,Electronics,5,1,1,N,Y,Overall pleased with the item,Works well. Bass is somewhat lacking but is pr...,2015-08-31


## 1. Imports and paths

In [19]:
import os
import re
import html
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

books_path = "/content/drive/MyDrive/Colab Notebooks/amazon_reviews_us_Books_v1_02.tsv"
electronics_path = "/content/drive/MyDrive/Colab Notebooks/amazon_reviews_us_Electronics_v1_00.tsv"

MODEL_DIR = "/content/drive/MyDrive/ReviewLens/models_prototype"
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42

## 2. Text cleaning and chunked sampling

In [20]:
def clean_review_text(headline, body):
    headline = "" if pd.isna(headline) else str(headline)
    body = "" if pd.isna(body) else str(body)

    text = f"{headline} {body}"
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)              # remove HTML
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"\s+", " ", text).strip().lower()  # normalize spaces

    return text


def sample_reviews(path, source_name, target_size, known_total_rows,
                   chunksize=100_000, random_state=42):
    columns = [
        "product_category", "star_rating", "helpful_votes", "total_votes",
        "review_headline", "review_body"
    ]

    rng = np.random.default_rng(random_state)
    collected = []

    # Collect extra candidates so filtering empty/very short reviews
    # still leaves enough records for the final random sample.
    keep_probability = min(1.0, (target_size * 1.20) / known_total_rows)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=columns,
        chunksize=chunksize,
        on_bad_lines="skip",
        low_memory=False
    ):
        chunk["review_text"] = [
            clean_review_text(headline, body)
            for headline, body in zip(chunk["review_headline"], chunk["review_body"])
        ]

        chunk = chunk[chunk["review_text"].str.len() >= 15].copy()
        keep = rng.random(len(chunk)) < keep_probability
        collected.append(chunk.loc[keep])

    candidates = pd.concat(collected, ignore_index=True)

    if len(candidates) < target_size:
        print(f"Warning: only {len(candidates):,} usable reviews found for {source_name}.")
        target_size = len(candidates)

    result = candidates.sample(n=target_size, random_state=random_state).copy()
    result["source_dataset"] = source_name
    return result


SAMPLE_PER_CATEGORY = 50_000

books_sample = sample_reviews(
    books_path,
    source_name="Books",
    target_size=SAMPLE_PER_CATEGORY,
    known_total_rows=3_105_371,
    random_state=RANDOM_STATE
)

electronics_sample = sample_reviews(
    electronics_path,
    source_name="Electronics",
    target_size=SAMPLE_PER_CATEGORY,
    known_total_rows=3_091_103,
    random_state=RANDOM_STATE + 1
)

reviews = pd.concat([books_sample, electronics_sample], ignore_index=True)

reviews["star_rating"] = pd.to_numeric(reviews["star_rating"], errors="coerce")
reviews["helpful_votes"] = pd.to_numeric(reviews["helpful_votes"], errors="coerce").fillna(0)
reviews["total_votes"] = pd.to_numeric(reviews["total_votes"], errors="coerce").fillna(0)
reviews["text_length"] = reviews["review_text"].str.len()

print(reviews["source_dataset"].value_counts())
print(f"\nTotal prototype reviews: {len(reviews):,}")
reviews[["source_dataset", "review_text", "text_length"]].head()

source_dataset
Books          50000
Electronics    50000
Name: count, dtype: int64

Total prototype reviews: 100,000


,source_dataset,review_text,text_length
0,Books,the gate behind the wall: a pilgrimage to jeru...,514
1,Books,grossly inflated book! would have made a great...,362
2,Books,a proper name : statistics summary and questio...,488
3,Books,"brotherhood of iron review \\""brotherhood of i...",2343
4,Books,not worth the paper it's printed on!! i didn't...,1686


## 3. TF-IDF → SVD

In [28]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, StandardScaler, normalize
from scipy.sparse import hstack, csr_matrix

# Keep negation words: "not good" must not become "good".
custom_stop_words = sorted(
    set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"}
)

# -------------------------------------------------------------------
# A. Identify terms strongly linked to only one product category.
#    Examples may include "novel", "author", "battery", "charger".
# -------------------------------------------------------------------
term_finder = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    max_features=80_000,
    stop_words=custom_stop_words
)

X_term_counts = term_finder.fit_transform(reviews["review_text"])
candidate_terms = term_finder.get_feature_names_out()

books_mask = reviews["source_dataset"].eq("Books").to_numpy()
electronics_mask = reviews["source_dataset"].eq("Electronics").to_numpy()

books_document_frequency = (
    np.asarray(X_term_counts[books_mask].getnnz(axis=0)).ravel()
    / books_mask.sum()
)

electronics_document_frequency = (
    np.asarray(X_term_counts[electronics_mask].getnnz(axis=0)).ravel()
    / electronics_mask.sum()
)

highest_frequency = np.maximum(
    books_document_frequency,
    electronics_document_frequency
)

category_dominance = highest_frequency / (
    np.minimum(books_document_frequency, electronics_document_frequency) + 1e-6
)

# Remove terms appearing in at least 0.2% of one category
# and at least five times more often in that category than the other.
topic_term_mask = (
    (highest_frequency >= 0.002)
    & (category_dominance >= 5)
)

topic_terms_removed = set(candidate_terms[topic_term_mask])
retained_terms = [
    term for term in candidate_terms
    if term not in topic_terms_removed
]

print(f"Candidate text terms: {len(candidate_terms):,}")
print(f"Category-specific terms removed: {len(topic_terms_removed):,}")
print(f"Retained text terms: {len(retained_terms):,}")

del X_term_counts


# -------------------------------------------------------------------
# B. Create writing-style features that work across all categories.
# -------------------------------------------------------------------
style_feature_names = [
    "log_word_count",
    "sentence_count",
    "vocabulary_diversity",
    "numeric_detail_rate",
    "first_person_rate",
    "long_term_experience_rate",
    "comparison_rate",
    "contrast_pros_cons_rate",
    "exclamation_count"
]

def build_style_features(texts):
    text_series = pd.Series(texts, dtype="string").fillna("").str.lower()

    tokens = text_series.str.findall(r"[a-z]+")
    word_count = tokens.str.len().clip(lower=1)

    vocabulary_diversity = tokens.apply(
        lambda words: len(set(words)) / max(len(words), 1)
    )

    return pd.DataFrame({
        "log_word_count": np.log1p(word_count),
        "sentence_count": text_series.str.count(r"[.!?]+").clip(lower=1),
        "vocabulary_diversity": vocabulary_diversity,
        "numeric_detail_rate": text_series.str.count(r"\d+") / word_count,
        "first_person_rate": (
            text_series.str.count(r"\b(i|my|mine|we|our|ours)\b") / word_count
        ),
        "long_term_experience_rate": (
            text_series.str.count(
                r"\b(after|since|days?|weeks?|months?|years?|used|using)\b"
            ) / word_count
        ),
        "comparison_rate": (
            text_series.str.count(
                r"\b(better|worse|than|compared|comparison|instead)\b"
            ) / word_count
        ),
        "contrast_pros_cons_rate": (
            text_series.str.count(
                r"\b(but|however|although|pros|cons|despite|except)\b"
            ) / word_count
        ),
        "exclamation_count": text_series.str.count(r"!")
    }).astype(np.float32)


# -------------------------------------------------------------------
# C. TF-IDF text features + writing-style features -> SVD.
# -------------------------------------------------------------------
filtered_vocabulary = {
    term: index for index, term in enumerate(retained_terms)
}

vectorizer = TfidfVectorizer(
    vocabulary=filtered_vocabulary,
    sublinear_tf=True,
    strip_accents="unicode",
    dtype=np.float32
)

X_text = vectorizer.fit_transform(reviews["review_text"])

style_features = build_style_features(reviews["review_text"])
style_scaler = StandardScaler()

X_style = style_scaler.fit_transform(style_features)

# Text receives weight 1.0; universal writing-style patterns receive 0.8.
STYLE_WEIGHT = 0.8
X_style = normalize(X_style) * STYLE_WEIGHT

X_combined = hstack(
    [X_text, csr_matrix(X_style)],
    format="csr"
)

svd = TruncatedSVD(
    n_components=100,
    n_iter=7,
    random_state=RANDOM_STATE
)

X_svd = svd.fit_transform(X_combined)

normalizer = Normalizer(copy=False)
X_reduced = normalizer.fit_transform(X_svd)

print("Text-feature shape:", X_text.shape)
print("Combined-feature shape:", X_combined.shape)
print("SVD shape:", X_reduced.shape)
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.2%}")

Candidate text terms: 80,000
Category-specific terms removed: 4,052
Retained text terms: 75,948
Text-feature shape: (100000, 75948)
Combined-feature shape: (100000, 75957)
SVD shape: (100000, 100)
Explained variance: 50.30%


## 4. Test several values of K

In [29]:
candidate_ks = range(3, 9)

rng = np.random.default_rng(RANDOM_STATE)
metric_indices = rng.choice(
    len(reviews),
    size=min(5_000, len(reviews)),
    replace=False
)

X_metric = X_reduced[metric_indices]
results = []

for k in candidate_ks:
    model = MiniBatchKMeans(
        n_clusters=k,
        batch_size=2048,
        n_init=5,
        max_iter=100,
        random_state=RANDOM_STATE
    )

    labels = model.fit_predict(X_reduced)
    metric_labels = labels[metric_indices]

    cluster_counts = np.bincount(labels, minlength=k)

    results.append({
        "k": k,
        "silhouette_higher_is_better": silhouette_score(X_metric, metric_labels),
        "davies_bouldin_lower_is_better": davies_bouldin_score(X_metric, metric_labels),
        "smallest_cluster_percent": cluster_counts.min() / len(labels) * 100,
        "largest_cluster_percent": cluster_counts.max() / len(labels) * 100
    })

metrics_df = pd.DataFrame(results).sort_values("silhouette_higher_is_better", ascending=False)
metrics_df

,k,silhouette_higher_is_better,davies_bouldin_lower_is_better,smallest_cluster_percent,largest_cluster_percent
2,5,0.178499,1.965406,13.999,30.426
1,4,0.175059,1.977906,16.121,31.095
0,3,0.164962,2.170214,30.673,35.134
3,6,0.153833,1.959196,12.723,25.654
5,8,0.145947,1.881868,7.932,17.643
4,7,0.116528,2.240249,11.962,18.962


## 5. Train the final prototype model

In [30]:
BEST_K = 5  # Replace this after reviewing metrics_df and cluster examples.

kmeans = MiniBatchKMeans(
    n_clusters=BEST_K,
    batch_size=2048,
    n_init=10,
    max_iter=200,
    random_state=RANDOM_STATE
)

reviews["cluster"] = kmeans.fit_predict(X_reduced)
reviews["cluster"].value_counts().sort_index()

,count
cluster,
0,14659
1,30236
2,23227
3,14509
4,17369


## 6. Inspect cluster terms and representative reviews

In [31]:
text_feature_count = X_text.shape[1]
feature_names = vectorizer.get_feature_names_out()

def get_top_terms(cluster_id, n_terms=12):
    reconstructed_centre = kmeans.cluster_centers_[cluster_id] @ svd.components_
    text_part = reconstructed_centre[:text_feature_count]

    top_indices = text_part.argsort()[-n_terms:][::-1]
    return feature_names[top_indices].tolist()


def get_top_style_signals(cluster_id, n_signals=4):
    reconstructed_centre = kmeans.cluster_centers_[cluster_id] @ svd.components_
    style_part = reconstructed_centre[text_feature_count:]

    top_indices = style_part.argsort()[-n_signals:][::-1]
    return [style_feature_names[index] for index in top_indices]


for cluster_id in range(BEST_K):
    print(f"\n{'=' * 70}")
    print(f"CLUSTER {cluster_id}")
    print("Top text patterns:", ", ".join(get_top_terms(cluster_id)))
    print("Top writing signals:", ", ".join(get_top_style_signals(cluster_id)))

    cluster_indices = np.where(reviews["cluster"].to_numpy() == cluster_id)[0]

    distances = np.linalg.norm(
        X_reduced[cluster_indices] - kmeans.cluster_centers_[cluster_id],
        axis=1
    )

    representative_indices = cluster_indices[np.argsort(distances)[:5]]

    display(
        reviews.iloc[representative_indices][
            ["source_dataset", "star_rating", "review_text"]
        ]
    )


CLUSTER 0
Top text patterns: not, good, great, like, works, just, little, work, no, really, does, don
Top writing signals: contrast_pros_cons_rate, vocabulary_diversity, log_word_count, comparison_rate


,source_dataset,star_rating,review_text
26590,Books,5.0,grownups will like it this coming-of-age book ...
87375,Electronics,5.0,ideal for getting prairie home companion out i...
21898,Books,5.0,you can't beat this book when it comes to pric...
39464,Books,3.0,too wordy although this book has an interestin...
30915,Books,2.0,it's not lonesome dove a weary tale with some ...



CLUSTER 1
Top text patterns: not, like, just, good, time, no, great, way, really, don, new, does
Top writing signals: log_word_count, sentence_count, exclamation_count, comparison_rate


,source_dataset,star_rating,review_text
18373,Books,5.0,"very good book, though scudder is slowing down..."
44441,Books,5.0,"back to the primitive \\""the call of the wild\..."
36879,Books,5.0,a double edged tale it is not by accident that...
23302,Books,5.0,the beauties of boswell quite a while back i p...
36545,Books,4.0,"a rebuke to denver's ""feminazi"" comment since ..."



CLUSTER 2
Top text patterns: stars, great, good, works, excellent, not, easy, work, nice, just, love, perfect
Top writing signals: vocabulary_diversity, exclamation_count, comparison_rate, numeric_detail_rate


,source_dataset,star_rating,review_text
16255,Books,5.0,great for the coffee table or the tackle box w...
1027,Books,4.0,well organised this book is well organised and...
39620,Books,1.0,quack risks doing serious harm to patients wit...
80955,Electronics,5.0,you buys this when you can it's a great why to...
39870,Books,5.0,straight talk and humor a very informative boo...



CLUSTER 3
Top text patterns: great, not, used, good, works, years, using, use, months, better, just, bought
Top writing signals: long_term_experience_rate, numeric_detail_rate, first_person_rate, vocabulary_diversity


,source_dataset,star_rating,review_text
40426,Books,5.0,"i followed it, now i'm retired at 42 years of ..."
67088,Electronics,3.0,every week i burn about 15 dvd's of church ser...
51824,Electronics,5.0,the fastest and best lasting rechargable batte...
76718,Electronics,3.0,"so so some of them went dead after 5-6 days, a..."
51130,Electronics,1.0,product worn out and possible fire hazard afte...



CLUSTER 4
Top text patterns: great, not, love, good, just, like, works, recommend, bought, use, stars, really
Top writing signals: first_person_rate, vocabulary_diversity, exclamation_count, comparison_rate


,source_dataset,star_rating,review_text
80446,Electronics,5.0,why we liked these! it met our needs for putti...
40001,Books,4.0,full of knowledge my fiance and i have decided...
23314,Books,4.0,beautiful parable this is a beauty parable for...
65104,Electronics,1.0,it was a great product until... it keeps blink...
32799,Books,5.0,the anger workbook i found this book to be ver...


## 7. Secondary helpful-vote validation

In [33]:
reviews["helpfulness_ratio"] = np.where(
    reviews["total_votes"] > 0,
    reviews["helpful_votes"] / reviews["total_votes"],
    np.nan
)

secondary_validation = (
    reviews.loc[reviews["total_votes"] >= 5]
    .groupby(["source_dataset", "cluster"])
    .agg(
        reviews_with_votes=("review_text", "size"),
        mean_helpfulness_ratio=("helpfulness_ratio", "mean"),
        median_total_votes=("total_votes", "median"),
        mean_text_length=("text_length", "mean")
    )
    .round(3)
)

secondary_validation

pd.crosstab(
    reviews["cluster"],
    reviews["source_dataset"],
    normalize="index"
).round(2)

source_dataset,Books,Electronics
cluster,,
0,0.49,0.51
1,0.79,0.21
2,0.38,0.62
3,0.25,0.75
4,0.38,0.62


## 8. Save only inference artifacts

In [35]:
joblib.dump(vectorizer, f"{MODEL_DIR}/tfidf_vectorizer.joblib", compress=3)
joblib.dump(svd, f"{MODEL_DIR}/svd_model.joblib", compress=3)
joblib.dump(normalizer, f"{MODEL_DIR}/normalizer.joblib", compress=3)
joblib.dump(kmeans, f"{MODEL_DIR}/clustering_model.joblib", compress=3)
joblib.dump(style_scaler, f"{MODEL_DIR}/style_scaler.joblib", compress=3)

cluster_profiles = {
    "0": {
        "label": "Balanced Evaluative Review",
        "description": "A review that discusses an overall opinion while weighing positive and negative points.",
        "informational_value": "Moderate to high",
        "top_terms": get_top_terms(0)
    },
    "1": {
        "label": "Extended Analytical Review",
        "description": "A longer, multi-sentence review with detailed explanation, reflection, or analysis.",
        "informational_value": "High",
        "top_terms": get_top_terms(1)
    },
    "2": {
        "label": "Concise General Opinion",
        "description": "A brief review centred on broad praise, criticism, ratings, or general impressions.",
        "informational_value": "Low to moderate",
        "top_terms": get_top_terms(2)
    },
    "3": {
        "label": "Long-Term Usage Experience",
        "description": "A review containing evidence from real use over time, often including duration, performance details, or specific issues.",
        "informational_value": "High",
        "top_terms": get_top_terms(3)
    },
    "4": {
        "label": "Personal Experience and Recommendation",
        "description": "A first-person review describing individual use, satisfaction, and recommendation behaviour.",
        "informational_value": "Moderate",
        "top_terms": get_top_terms(4)
    }
}

with open(f"{MODEL_DIR}/cluster_profiles.json", "w", encoding="utf-8") as file:
    json.dump(cluster_profiles, file, indent=2, ensure_ascii=False)

metadata = {
    "pipeline": (
        "clean text -> category-term-reduced TF-IDF + "
        "universal writing-style features -> TruncatedSVD -> "
        "Normalizer -> MiniBatchKMeans"
    ),
    "sample_size": int(len(reviews)),
    "sources": ["Books", "Electronics"],
    "n_clusters": int(BEST_K),
    "svd_components": 100,
    "retained_text_features": int(X_text.shape[1]),
    "style_features": style_feature_names,
    "style_weight": STYLE_WEIGHT,
    "category_specific_terms_removed": int(len(topic_terms_removed)),
    "random_state": RANDOM_STATE
}
with open(f"{MODEL_DIR}/model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved model artifacts to:", MODEL_DIR)

Saved model artifacts to: /content/drive/MyDrive/ReviewLens/models_prototype


## 9. Test fast inference

In [36]:
def transform_reviews_for_model(review_texts):
    cleaned_texts = [
        clean_review_text("", text)
        for text in review_texts
    ]

    # Same filtered TF-IDF transformation used during training.
    X_text_new = vectorizer.transform(cleaned_texts)

    # Same writing-style features and scaling used during training.
    style_new = build_style_features(cleaned_texts)
    X_style_new = style_scaler.transform(style_new)
    X_style_new = normalize(X_style_new) * STYLE_WEIGHT

    # Recreate the exact combined feature representation.
    X_combined_new = hstack(
        [X_text_new, csr_matrix(X_style_new)],
        format="csr"
    )

    X_svd_new = svd.transform(X_combined_new)
    return normalizer.transform(X_svd_new)


def analyze_review(review_text):
    if not isinstance(review_text, str) or not review_text.strip():
        raise ValueError("Please enter a non-empty review.")

    X_new = transform_reviews_for_model([review_text])

    distances = kmeans.transform(X_new)[0]
    ranked_clusters = np.argsort(distances)

    nearest_cluster = int(ranked_clusters[0])
    alternative_cluster = int(ranked_clusters[1])
    profile = cluster_profiles[str(nearest_cluster)]

    return {
        "review_pattern": profile["label"],
        "description": profile["description"],
        "informational_value": profile["informational_value"],
        "closest_cluster": nearest_cluster,
        "alternative_pattern": cluster_profiles[str(alternative_cluster)]["label"],
        "distance_to_closest_cluster": round(float(distances[nearest_cluster]), 4),
        "note": (
            "This is the closest learned review pattern, not an objective "
            "or guaranteed usefulness judgement."
        )
    }


analyze_review(
    "I have used these headphones for six months. "
    "The sound is excellent and the battery lasts around eight hours, "
    "but the earcups become uncomfortable after long use."
)

{'review_pattern': 'Long-Term Usage Experience',
 'description': 'A review containing evidence from real use over time, often including duration, performance details, or specific issues.',
 'informational_value': 'High',
 'closest_cluster': 3,
 'alternative_pattern': 'Balanced Evaluative Review',
 'distance_to_closest_cluster': 0.8391,
 'note': 'This is the closest learned review pattern, not an objective or guaranteed usefulness judgement.'}